# FLUX Phase 1.5: Causal Wave Chaining (CWC)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Add causal direction, contradiction detection, and implication chaining to the FLUX architecture. This phase extends the semantic encoder with reasoning arrows, enabling order sensitivity, contradiction tension, and implication propagation.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phase 2** — Verify RFC checkpoint loads and works
3. **Smoke Test** — Verify CausalWaveChainer builds, encodes, and extends
4. **Train** — Causal coherence, order, contradiction, implication (3000 steps)
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — Order sensitivity (original > shuffled)
7. **Test 2** — Contradiction detection (tension)
8. **Test 3** — Implication propagation (reasoning chains)
9. **Demo 1** — Order sensitivity visualization
10. **Demo 2** — Contradiction tension heatmap
11. **Demo 3** — Implication chain tracing
12. **Results** — Generate RESULTS_PHASE_1_5.md
13. **Final** — Upload logs → HuggingFace & push to GitHub

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)


---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")
    result = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=str(WORK_DIR), capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print("✓ Pulled latest")
else:
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform
from pathlib import Path

# Add project paths
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1_5'))

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults,
)

# Init logger
log = PhaseLogger(phase=1.5)
log.separator("Phase 1.5: Causal Wave Chaining")

# Hardware
device = get_device()
hw = get_hardware_info()
log.cell_start("Cell 3 — Hardware & Secrets")
for k, v in hw.items():
    log.info(f"{k}: {v}")
    print(f"  {k}: {v}")

# HF Token
token = _resolve_hf_token()
if token:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token found")
    print("  ⚠ No HF token — uploads will be skipped")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phase 2 Checkpoint & Smoke Test

In [ ]:
log.cell_start("Cell 4 — Load Phase 2 & Smoke Test")

# Always load Phase 2 model from HuggingFace Hub (not local checkpoint)
# Model URL: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)
from flux_utils import load_checkpoint
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from cse import ContinuousSemanticEncoder

# Load Phase 2 checkpoint (RFC)
checkpoint_p2 = load_checkpoint(2)
field_cfg = checkpoint_p2['config']['field']
cse_cfg = checkpoint_p2['config']['cse']

# Build models
cse = ContinuousSemanticEncoder(**cse_cfg)
cse.load_state_dict(checkpoint_p2['cse_state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False

field = ResonanceField(**field_cfg).to(device)
field.load_state_dict(checkpoint_p2['state_dict'])
field = field.eval()

log.success(f"CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")
log.success(f"ResonanceField loaded: {sum(p.numel() for p in field.parameters()):,} params")

# Smoke test CSE
wave = cse.encode("The causal wave awakens")
assert wave.full.shape[-1] == 432, f"Bad wave dim: {wave.full.shape}"
assert not torch.isnan(wave.full).any(), "NaN in wave!"
log.success(f"CSE smoke test passed: wave shape {wave.full.shape}")

# Smoke test ResonanceField
vec = wave.full.mean(dim=0)
loc = field.perturb(vec)
feats, sims, locs = field.query(vec)
assert feats.shape[1] == field_cfg['features']
log.success(f"ResonanceField smoke: perturb→({loc.h},{loc.w},{loc.d}), query→{feats.shape}")

del field
print("  ✓ Phase 2 RFC loaded and verified")
print("  ✓ CSE loaded and frozen")
print("  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
log.cell_end("Cell 4 — Load Phase 2 & Smoke Test", "PASS")

---
## Cell 5: Train Phase 1.5 Model

In [ ]:
log.cell_start("Cell 5 — Training")
import os
os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python train_cwc.py --device {device}
os.chdir(str(WORK_DIR))
log.cell_end("Cell 5 — Training", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

# Save checkpoint and upload to HuggingFace Hub
from flux_utils import save_checkpoint, upload_checkpoint_to_hf
from pathlib import Path

ckpt_path = Path('checkpoints/phase1_5.phase.pt')
if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / 1e6
    log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    uploaded = upload_checkpoint_to_hf(phase=1.5, hf_token=token)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
    else:
        log.warning("Checkpoint upload skipped — check HF_TOKEN")
else:
    log.error("Checkpoint NOT found — training may have failed")

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")